# Enterprise SaaS Churn Capstone: Adoption & Value Realization
**Data notice:** The companion CSV is synthetic/mock data, not real customer telemetry.
## Business question
Can account profile, adoption, engagement and value-realization signals identify accounts at risk of churn or downgrade?
Target: `churn` (1 = churned/downgraded; 0 = renewed). Primary metrics: recall and Average Precision (PR-AUC).

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, precision_score, average_precision_score, roc_auc_score, confusion_matrix, PrecisionRecallDisplay
RANDOM_STATE = 42
df = pd.read_csv('../data/saas_telemetry.csv')
print('Shape:', df.shape)
display(df.head())
display(df.isna().sum().to_frame('missing_count'))
display(df['churn'].value_counts(normalize=True).rename('share'))

## Data preparation and feature engineering
Exclude the account identifier and target from predictors. Derived signals provide normalized commercial value and adoption gaps.

In [ ]:
df['arr_per_seat'] = df['annual_recurring_revenue'] / df['seat_licenses'].replace(0, np.nan)
df['adoption_gap'] = 1 - df['license_utilization_pct']
df['value_deficit'] = 1 - df['kpi_milestones_achieved_pct']
target, id_col = 'churn', 'account_id'
X = df.drop(columns=[target, id_col])
y = df[target].astype(int)
print('Duplicate IDs:', df[id_col].duplicated().sum())
print('Churn prevalence:', round(y.mean(), 3))

## Exploratory data analysis
Group-level differences are descriptive associations, not causal effects.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
df.groupby('tier')['churn'].mean().plot(kind='bar', ax=axes[0], title='Churn rate by tier')
df['utilization_band'] = pd.cut(df['license_utilization_pct'], [0,.4,.6,.8,1.01], labels=['<40%','40–60%','60–80%','>80%'])
df.groupby('utilization_band', observed=True)['churn'].mean().plot(kind='bar', ax=axes[1], title='Churn by license utilization')
plt.tight_layout(); plt.show()
display(df.groupby('churn')[['license_utilization_pct','dau_mau_ratio','core_modules_active','kpi_milestones_achieved_pct','csat_score']].mean().T)

## Baseline model: Logistic Regression
Use a stratified 75/25 split and fit preprocessing only on the training partition. Balanced class weights support the minority class.

In [ ]:
numeric = X.select_dtypes(include='number').columns.tolist()
categorical = X.select_dtypes(exclude='number').columns.tolist()
numeric_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())])
categorical_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])
prep = ColumnTransformer([('num', numeric_pipe, numeric), ('cat', categorical_pipe, categorical)])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.25, random_state=RANDOM_STATE, stratify=y)
model = Pipeline([('prep', prep), ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE))])
model.fit(X_train, y_train)
pred = model.predict(X_test); score = model.predict_proba(X_test)[:, 1]
metrics = {'Recall': recall_score(y_test, pred), 'Precision': precision_score(y_test, pred), 'PR-AUC (Average Precision)': average_precision_score(y_test, score), 'ROC-AUC': roc_auc_score(y_test, score)}
display(pd.Series(metrics).to_frame('test_score'))

## Diagnostics and next steps
Review the confusion matrix and precision-recall trade-off. Tune the operating threshold against ARR at risk and Customer Success capacity. For production, use time-based validation, leakage controls, calibration, segment-level evaluation and human review.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=['Renewed','Churned'])
plt.title('Baseline confusion matrix'); plt.show()
PrecisionRecallDisplay.from_predictions(y_test, score)
plt.title('Precision-recall curve'); plt.show()